# 01 · scikit-learn 的世界觀

歡迎來到 **程式實驗室 → 機器學習 → scikit-learn**。

scikit-learn（簡稱 sklearn）是 Python 機器學習的標準工具箱。它最聰明的地方在於：**幾十種模型，全都長一個樣子**。只要學會一套 `fit` / `predict` / `transform` 的節奏，你換任何模型都不用重學。

這堂課不急著教演算法，先把這個**統一的世界觀**建立起來——之後每一課都是在這個骨架上長出來的。

> 💡 這是一個可執行的 notebook。每個程式碼格子按 ▶️（或 `Shift+Enter`）就會跑。放心改參數、亂玩——你改的是自己的副本，不會動到原檔。

## 學習目標

- 理解 sklearn 統一的 **estimator API**：`fit` / `predict` / `transform`
- 分清楚 **監督式** 與 **非監督式** 學習
- 用內建的鳶尾花資料集，跑出你的第一個分類器
- 知道為什麼要把資料切成 **訓練集 / 測試集**

## 1. 一個節奏走天下：fit / predict / transform

sklearn 裡每個模型都是一個 **estimator（估計器）** 物件，操作只有三個動詞：

| 方法 | 中文 | 做什麼 |
| --- | --- | --- |
| `model.fit(X, y)` | 訓練 | 給它特徵 `X` 和答案 `y`，讓它從資料中學 |
| `model.predict(X)` | 預測 | 對新的特徵 `X` 給出預測 |
| `model.transform(X)` | 轉換 | 把資料變形（前處理器才有，如標準化） |

換模型時，只有 `model = ...` 那一行要改，`fit` / `predict` 完全不動。這就是 sklearn 的威力——**API 一致，知識可遷移**。

## 2. 先認識資料：鳶尾花（iris）

我們用最經典的入門資料集：**鳶尾花**。150 朵花，每朵量了 4 個特徵（花萼/花瓣的長寬），目標是分辨它屬於 3 個品種中的哪一種。

In [1]:
from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data        # 特徵：150 列 × 4 欄
y = iris.target      # 答案：每朵花的品種編號 0/1/2

print("特徵形狀 X:", X.shape)
print("特徵名稱:", iris.feature_names)
print("品種名稱:", iris.target_names)
print("\n前 3 朵花的特徵:\n", X[:3])
print("前 3 朵花的答案:", y[:3])

特徵形狀 X: (150, 4)
特徵名稱: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
品種名稱: ['setosa' 'versicolor' 'virginica']

前 3 朵花的特徵:
 [[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]]
前 3 朵花的答案: [0 0 0]


**慣例**：sklearn 一律用大寫 `X` 表示特徵矩陣（二維：列是樣本、欄是特徵），小寫 `y` 表示目標（一維）。記住這個慣例，看任何 sklearn 程式碼都會順。

## 3. 切資料：訓練集 vs 測試集

關鍵原則：**不能用考過的題目評估學生**。如果拿訓練時看過的資料來打分數，模型只要「背答案」就能拿高分，但對新資料毫無用處。

所以我們先切一刀：一部分拿來 `fit`（訓練），另一部分藏起來，最後才拿來驗收。

In [2]:
from sklearn.model_selection import train_test_split

# test_size=0.25 → 留 25% 當測試集；random_state 固定隨機種子讓結果可重現
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("訓練集:", X_train.shape, "  測試集:", X_test.shape)

訓練集: (112, 4)   測試集: (38, 4)


> `stratify=y` 讓切分後三個品種的比例維持一致——分類問題幾乎都該加。

## 4. 你的第一個分類器：K 近鄰（KNN）

KNN 的想法直白到不可思議：**要預測一朵新花是什麼品種，就看離它最近的 K 朵花是什麼**，少數服從多數。

看它怎麼套進 `fit` / `predict` 的節奏：

In [3]:
from sklearn.neighbors import KNeighborsClassifier

# 1. 建立模型
model = KNeighborsClassifier(n_neighbors=5)

# 2. 訓練（只餵訓練集！）
model.fit(X_train, y_train)

# 3. 對測試集預測
y_pred = model.predict(X_test)

# 4. 打分數：預測對幾成？
accuracy = model.score(X_test, y_test)
print(f"測試集準確率：{accuracy:.1%}")
print("前 10 筆預測:", y_pred[:10])
print("前 10 筆答案:", y_test[:10])

測試集準確率：97.4%
前 10 筆預測: [0 1 1 1 0 1 2 2 2 2]
前 10 筆答案: [0 1 1 1 0 1 2 2 2 2]


就這樣，你訓練並評估了第一個機器學習模型。回頭看那四步——之後每一課、每一個模型，骨架都是這個：

```python
model = SomeEstimator(...)   # 換模型只改這行
model.fit(X_train, y_train)  # 訓練
model.predict(X_test)        # 預測
model.score(X_test, y_test)  # 評估
```

**動手試試**：把 `n_neighbors=5` 改成 `1` 或 `15`，重跑看準確率怎麼變。

## 5. 監督 vs 非監督

剛剛的鳶尾花有「答案 `y`」可學——這叫 **監督式學習（supervised）**，又分兩類：

- **分類（classification）**：預測類別（哪個品種）→ 本課的 KNN
- **迴歸（regression）**：預測連續數值（房價、溫度）→ 第 03 課

如果資料**沒有答案**，只能靠資料本身的結構找規律（自動分群、降維），那叫 **非監督式學習（unsupervised）**→ 第 06 課。

## 小結

- sklearn 每個模型都是 estimator，操作就 `fit` / `predict` / `transform` 三招。
- 特徵叫 `X`（二維）、答案叫 `y`（一維），是不成文慣例。
- 一定要切 **訓練 / 測試集**，用沒看過的資料評估才誠實。
- 換模型只改建立物件那一行，其餘流程不變。

## 練習

1. 把 KNN 換成 `from sklearn.linear_model import LogisticRegression`，其餘程式碼**完全不改**，看是否照跑、準確率多少。（體會 API 一致的好處）
2. 試試 `test_size=0.5`，準確率有變化嗎？想想為什麼。

下一課，我們專心做好**分類**，並把模型的「決策邊界」畫出來看。